# 03 · Activations and vanishing gradients

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/03_activations_depth.ipynb) &nbsp; 📖 Concept deck: [Activation functions](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/02-activation-functions/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** sigmoid, tanh, ReLU, GELU and SiLU with their derivatives, and a measurement of how gradient size changes through a deep stack for each.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


from math import erf, sqrt
_erf = np.vectorize(erf)

## Task 1: Activations and their derivatives

Return a dict `name -> (f, df)` for `sigmoid`, `tanh`, `relu`, `gelu` (exact: $x\Phi(x)$ with $\Phi(x) = \frac{1}{2}(1 + \mathrm{erf}(x/\sqrt 2))$) and `silu` ($x\,\sigma(x)$).
Derivatives: $\sigma' = \sigma(1-\sigma)$; $\mathrm{GELU}' = \Phi(x) + x\varphi(x)$ with $\varphi(x) = e^{-x^2/2}/\sqrt{2\pi}$; $\mathrm{SiLU}' = \sigma(x)(1 + x(1 - \sigma(x)))$.

In [ ]:
def activations():
    # TODO: return {'sigmoid': (f, df), 'tanh': ..., 'relu': ..., 'gelu': ..., 'silu': ...}
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_activations(f):
    A = f(); x = np.linspace(-4, 4, 41)
    assert set(A) == {'sigmoid', 'tanh', 'relu', 'gelu', 'silu'}
    for k, (fn, df) in A.items():
        num = (fn(x + 1e-6) - fn(x - 1e-6)) / 2e-6
        m = np.abs(x) > 1e-3       # relu has a kink at 0
        assert np.allclose(df(x)[m], num[m], atol=1e-5), f"{k}: derivative mismatch"
    assert np.isclose(A['gelu'][0](np.array([1.0]))[0], 0.8413447, atol=1e-6)
    assert np.isclose(A['silu'][0](np.array([1.0]))[0], 1 / (1 + np.exp(-1)), atol=1e-9)

check("activations", _test_activations, activations)

## Task 2: Gradient size through depth

Build a stack of `depth` layers $h \leftarrow \text{act}(h W_\ell)$ of width `width`, with $W_\ell$ drawn with std $\sqrt{\text{gain}/\text{width}}$. Push a batch of 256 standard-normal inputs forward, then backpropagate a gradient of ones from the top.
Return the array of mean |gradient| arriving at each layer's input, from the top layer (index 0) down to the bottom (index depth−1).

In [ ]:
def grad_through_depth(name, depth=30, width=256, gain=1.0, seed=0):
    f, df = activations()[name]          # your Task 1 functions
    r = np.random.default_rng(seed)
    # TODO: forward (store pre-activations), backward, record mean |grad| per layer
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_grad_through_depth(f):
    g = f('sigmoid', depth=20)
    assert len(g) == 20
    assert g[-1] < 1e-6 * g[0], "sigmoid gradients should vanish by > 10^6 over 20 layers (σ' ≤ 1/4)"
    r = f('relu', depth=20, gain=2.0)
    assert 0.1 < r[-1] / r[0] < 10, "ReLU with He gain 2 should keep gradients roughly constant"

check("grad_through_depth", _test_grad_through_depth, grad_through_depth)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Activations and their derivatives
def activations_solution():
    sig = lambda x: 1 / (1 + np.exp(-x))
    Phi = lambda x: 0.5 * (1 + _erf(x / sqrt(2)))
    phi = lambda x: np.exp(-x * x / 2) / sqrt(2 * np.pi)
    return {'sigmoid': (sig, lambda x: sig(x) * (1 - sig(x))),
            'tanh': (np.tanh, lambda x: 1 - np.tanh(x) ** 2),
            'relu': (lambda x: np.maximum(x, 0), lambda x: (x > 0).astype(float)),
            'gelu': (lambda x: x * Phi(x), lambda x: Phi(x) + x * phi(x)),
            'silu': (lambda x: x * sig(x), lambda x: sig(x) * (1 + x * (1 - sig(x))))}

check("activations (reference)", _test_activations, activations_solution)

In [ ]:
# @title Solution for Task 2: Gradient size through depth
def grad_through_depth_solution(name, depth=30, width=256, gain=1.0, seed=0):
    f, df = activations_solution()[name]
    r = np.random.default_rng(seed)
    Ws = [r.normal(scale=np.sqrt(gain / width), size=(width, width)) for _ in range(depth)]
    h = r.normal(size=(256, width)); pre = []
    for W in Ws:
        z = h @ W; pre.append(z); h = f(z)
    g, out = np.ones_like(h), []
    for W, z in zip(reversed(Ws), reversed(pre)):
        g = (g * df(z)) @ W.T; out.append(np.abs(g).mean())
    return np.array(out)

check("grad_through_depth (reference)", _test_grad_through_depth, grad_through_depth_solution)

---
## Experiments
**Which activations keep the gradient alive?** Compare the ratio of bottom-layer to top-layer gradient size after 30 layers. For ReLU the variance-preserving gain is exactly 2 (He init). SiLU's outputs have a non-zero mean, so the simple rule breaks down; here a gain near 2.25 keeps gradients steady, and small changes either way make them shrink or grow by orders of magnitude over 30 layers. That sensitivity is one reason deep transformers lean on normalisation and residual connections instead of exact initialisation.

In [ ]:
for name, gain in [('sigmoid', 1.0), ('tanh', 1.0), ('relu', 2.0), ('gelu', 2.0), ('silu', 2.0), ('silu', 2.25), ('silu', 2.7)]:
    g = grad_through_depth_solution(name, depth=30, gain=gain)
    print(f"{name:8s} gain {gain}: bottom/top gradient ratio = {g[-1] / g[0]:.2e}")

---
## Takeaways
* Sigmoid's derivative is at most 1/4, so gradients shrink geometrically through depth.
* ReLU-family activations (ReLU, GELU, SiLU) pass gradients with slope ≈ 1 for positive inputs; with the right init scale they stay healthy.
* Modern LLMs use GELU or SiLU (inside SwiGLU) as smooth relatives of ReLU.